# Schizophrenia Pathway Classifier — Enrichment Analysis

## Objective

In this notebook I use enrichment analysis to see if immune/inflammatory pathway genes show expression differences in SCZ samples compared to control samples. To do this, I run differential expression testing then apply FDR correction, afterward I rank the gene list, then lastly use GSEA against the immune/inflammatory gene set. This answers the first part of my primary hypothesis: whether immune/inflammatory pathway genes, implicated in schizophrenia by independent studies, show enrichment in expression differences between SCZ vs. control subjects in this dataset.

## Input
- `../data/processed/merged_df.csv`

## 2.1 Setup & Load Data

Import most of the same libraries from EDA, with the addition of `scipy` for statistical testing, `statsmodels` for FDR, and `gseapy` for GSEA. Additionally, I set random state and absolute path to `merged_df.csv` for reproducibility. I read in the `merged_df.csv` since it contains both the expression values and metadata needed for the downstream analyses. 

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse 

from scipy import stats
import statsmodels.stats.multitest as smt
import gseapy as gp

sns.set_theme()
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_STATE = 42
PROCESSED_DATA_PATH = '../data/processed/merged_df.csv'

In [3]:
merged_df = pd.read_csv(PROCESSED_DATA_PATH)
# check shape and value counts to confirm df was imported correctly
print(merged_df.shape)
merged_df['diagnosis'].value_counts() 

(59, 30065)


diagnosis
schizophrenia    30
control          29
Name: count, dtype: int64

Correct shape (59, 30065) and class balance (30/29 SCZ samples vs. control)

## 2.2 Probe-to-Gene Mapping

I translate probe IDs to gene symbols, since the `merged_df`'s columns are currently Affymetrix probe IDs, so that I can compare immune/inflammatory pathway list against my ranked list created downstream for GSEA. To do this I load GPL570 using `GEOparse.get_GEO()`, and then use that to produce a lookup dictionary that maps probes to genes. 

To map probes to genes I first load `GPL570` using `GEOparse.get_GEO()`, then I confirm shape and look at the columns to identify which contains gene symbols. Once I've identified the correct column containing gene symbols, I check for any duplicates or missing values, since multiple probes (duplicates) can map to the same gene and some probes may have no gene annotation (missing values) and therefore can't be tested against a gene set. 

In [9]:
gpl = GEOparse.get_GEO(geo="GPL570", destdir="../data/raw/")
print(gpl.table.shape)
print(gpl.table.columns.tolist())
print('duplicate gene symbols:', gpl.table['Gene Symbol'].duplicated().sum())
print('missing gene symbols:', gpl.table['Gene Symbol'].isna().sum())


20-Aug-2026 13:38:36 DEBUG utils - Directory ../data/raw/ already exists. Skipping.
20-Aug-2026 13:38:36 INFO GEOparse - File already exist: using local version.
20-Aug-2026 13:38:36 INFO GEOparse - Parsing ../data/raw/GPL570.txt: 
20-Aug-2026 13:38:36 DEBUG GEOparse - PLATFORM: GPL570


(54675, 16)
['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title', 'Gene Symbol', 'ENTREZ_GENE_ID', 'RefSeq Transcript ID', 'Gene Ontology Biological Process', 'Gene Ontology Cellular Component', 'Gene Ontology Molecular Function']
duplicate gene symbols: 31154
missing gene symbols: 8893


/Users/joshuasim/Desktop/summer_projects/schizophrenia-pathway-classifier/venv/lib/python3.12/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (0: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


Shape is (54675,16) matching typical GPL570 structure. I identified `Gene Symbol` as the column containing gene symbols, and found 31154 duplicate gene symbols and 8893 missing gene symbols. The high duplicate count is expected since multiple probes per gene is common on Affymetrix arrays — not a data quality issue. The missing gene symbols are likely control probes, ESTs, and other unannotated sequences that are included in the array design, and fall within a normal range for GPL570. 

To create a look-up table mapping probe IDs to gene symbols, I will set `gpl_table`'s index to `ID`, then convert the `Gene Symbol` column into a dictionary using `.to_dict()`, giving probe ID to gene symbol pairs. 

In [11]:
gpl_key = gpl.table.set_index('ID')
probe_to_gene = gpl_key['Gene Symbol'].to_dict()

# check mapping with probe ID 224264_x_at (outliers in EDA traced back to this probe)
probe_to_gene['224264_x_at']

'ZAN'

I verify the mapping was correctly done by checking probe `224264_x_at` against what gene symbol it is officially associated to (ZAN, according to biogps) — since `probe_to_gene` returned the correct symbol `ZAN`, it confirms the mapping is accurate. 

## 2.3 Outlier Probe Flag
Since the outlier samples' diagnosis split was ambiguous (5 of the 6 outlier samples were SCZ, not enough to come to a conclusion), I decided to keep probe `224264_x_at`, letting differential expression testing itself determine whether the outliers are statistically significant.

In [12]:
flagged_probes = ['224264_x_at']
# this will be used in DE testing to flag outliers

## 2.4 Differential Expression Testing

## 2.5 Multiple Testing Correction

## 2.6 DE Results Summary

## 2.7 Rank Gene List for GSEA

## 2.8 GSEA Against Immune/Inflammatory Gene Set

## 2.9 Hypothesis 1 answering

## 2.10 Summary